# 嘉音 /a/ の等フォルマント解の大域探索
CPUランタイムで上から順に実行。6温度×3シード。成分数は近傍グラフと経路の離散評価に依存する推定値。
GUDHIで近傍グラフのクリーク複体を3次元まで構成し、H1・H2を計算。三角形・四面体の内部のフォルマント誤差は未測定。

In [ ]:
%%bash
set -eu
cd /content
test -d kanon-vocal-lab-2 || git clone --depth 1 --branch colab https://github.com/k0tokay/kanon-vocal-lab-2.git
cd kanon-vocal-lab-2
git pull --ff-only origin colab
apt-get -qq update
apt-get -qq install -y libopenal-dev
g++ -Wl,-z,defs -shared -fPIC -std=c++14 -O2 -DHAVE_OPENAL -Ibackend measure.cpp backend/*.cpp -lopenal -pthread -o measure.so

In [ ]:
%cd /content/kanon-vocal-lab-2
%run explore.py

In [ ]:
%pip -q install gudhi
import gudhi
import matplotlib.pyplot as plt
import csv

# Flag filtration on the same five-neighbour graph; each clique enters
# at the largest of its edge weights. Tetrahedra allow H2 classes to die.
cutoff = 0.05
tree_ph = gudhi.SimplexTree()
for i, e in enumerate(errors):
    if e <= cutoff:
        tree_ph.insert([i], filtration=float(e))
for i, j, e in paths:
    if e <= cutoff:
        tree_ph.insert([i, j], filtration=float(e))
tree_ph.expansion(3)
tree_ph.persistence(homology_coeff_field=2, persistence_dim_max=True)
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
with open('higher-bars.csv', 'w', newline='') as stream:
    writer = csv.writer(stream)
    writer.writerow(['dimension', 'birth', 'death'])
    for d, ax in zip([1, 2], axes):
        bars = sorted(tree_ph.persistence_intervals_in_dimension(d), key=lambda b: b[0])
        counts = [sum(b <= t < end for b, end in bars) for t in (.005,.01,.02,.05)]
        print('H', d, 'bars', len(bars), 'Betti at thresholds', counts, flush=True)
        for row, (birth, death) in enumerate(bars):
            writer.writerow([d, birth, death])
            end = min(death, cutoff)
            ax.hlines(row, birth, end, color='steelblue')
            if np.isinf(death):
                ax.plot(end, row, '>', color='steelblue', clip_on=False)
        ax.set(title=f'H{d} ({len(bars)} intervals)', xlabel='Maximum absolute log-formant error', xlim=(0,cutoff), ylabel='Interval')
fig.savefig('higher-bars.png', dpi=160)
plt.show()

In [ ]:
from google.colab import files
files.download('samples.npy')